Importing the Json file from the data cleaning notebook

In [3]:
import os
import json

with open('celebrity_file_names_dict.json', 'r') as f:
    names_dict = json.load(f)

Assiging unique numbers to players

In [4]:
count = 0
names_scale = {}
for names in names_dict.keys():
    names_scale[names] = count
    count += 1
names_scale

{'charles leclerc': 0,
 'dhoni': 1,
 'lewis hamilton': 2,
 'lionel messi': 3,
 'max verstappen': 4,
 'neymar': 5,
 'rohit sharma': 6,
 'ronaldo': 7,
 'virat kohli': 8}

Transform the images using wavelet transfrom 

In [5]:
import numpy as np
import cv2
import pywt

def wavelet_transform(image, wavelet='haar', level=1):
    img_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    img_gray = np.float32(img_gray) / 255.0

    coeffs_h = pywt.wavedec2(img_gray, wavelet, level=level)
    
    coeffs = list(coeffs_h)
    coeffs[0] *= 0

    img_reconstructed = pywt.waverec2(coeffs, wavelet)

    img_reconstructed = np.clip(img_reconstructed, 0, 1)

    return (img_reconstructed * 255).astype(np.uint8)
    

Create X and y features from the transformed image

In [6]:
X , y = [], []
for names, files in names_dict.items():
    for file in files:
        img = cv2.imread(file)
        if img is None:
            continue
        wavelet_img = wavelet_transform(img)
        scaled_img_raw = cv2.resize(img, (32, 32))
        scaled_img_wavelet = cv2.resize(wavelet_img, (32, 32))
        combined_img = np.vstack((scaled_img_raw.reshape(32*32*3, 1), scaled_img_wavelet.reshape(32*32, 1)))
        X.append(combined_img)
        y.append(names_scale[names])

Convert the python list to numpy array

In [7]:
X = np.array(X).reshape(len(X), 32*32*3 + 32*32).astype(float)

Import the required libs for model traning

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

Split the dataset to train and test

In [9]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Create a dict to perform Gridsearchcv

In [49]:
models = {
    'RandomForest': (
        RandomForestClassifier(),
        {
            'classifier__n_estimators': [100, 150, 200],
            'classifier__max_depth': [None, 10, 20]
        }
    ),

    'SVM': (
        SVC(),
        {
            'classifier__C': [1, 10, 100],
            'classifier__kernel': ['linear', 'rbf']
        }
    ),

    'LogisticRegression': (
        LogisticRegression(),
        {
            'classifier__max_iter': [500, 1000]
        }
    )
}

Use gridsearchcv to find best model with best parameters

In [50]:
results = []
best_model = {}

for name, (model, params) in models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])
    grid = GridSearchCV(
        pipeline,
        params,
        cv=5,
        return_train_score=False
    )
    grid.fit(x_train, y_train)
    results.append({
        'model': name,
        'best_score': grid.best_score_,
        'best_params': grid.best_params_
    })
    best_model[name] = grid.best_estimator_
results_df = pd.DataFrame(results)
results_df
best_model

{'RandomForest': Pipeline(steps=[('scaler', StandardScaler()),
                 ('classifier', RandomForestClassifier(n_estimators=150))]),
 'SVM': Pipeline(steps=[('scaler', StandardScaler()),
                 ('classifier', SVC(C=1, kernel='linear'))]),
 'LogisticRegression': Pipeline(steps=[('scaler', StandardScaler()),
                 ('classifier', LogisticRegression(max_iter=500))])}

In [52]:
results_df

,model,best_score,best_params
0,RandomForest,0.564444,"{'classifier__max_depth': None, 'classifier__n..."
1,SVM,0.652754,"{'classifier__C': 1, 'classifier__kernel': 'li..."
2,LogisticRegression,0.701063,{'classifier__max_iter': 500}


In [37]:
best_model['LogisticRegression'].score(x_test, y_test)

0.7543859649122807

In [53]:
best_model['SVM'].score(x_test, y_test)

0.7719298245614035

In [54]:
best_model['RandomForest'].score(x_test, y_test)

0.6140350877192983

In [ ]:
model = best_model['SVM']
model.fit(x_train, y_train)


0.7543859649122807

Import the model using joblib

In [55]:
import joblib
joblib.dump(model, 'celebrity_classification_model.pkl')

['celebrity_classification_model.pkl']

Import names scale using json

In [56]:
import json
with open('names_scale.json', 'w') as f:
    json.dump(names_scale, f)